# S3Eval: Evaluation Pipeline

## Environment Setup

In [ ]:
import sys
import os
import re
import json
import time
import random as _rng
try:
    import resource  # Unix-only
except ImportError:
    resource = None
import platform
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI
from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa, aggregate_raters

from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider

# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, 'resources', 'data', 'mutated_bad_PDDL_AP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")

def get_device_info():
    """Return device info dict."""
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }


In [ ]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')

In [ ]:
def model_short_name(model_name):
    """Generate a short readable name from a full model identifier.
    """
    name = model_name.split("/")[-1]  # strip org prefix
    for suffix in ["-instruct", "-Instruct", "-chat", "-Chat"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_bad_dataset(bad_pddl_dir, cve_descriptions):
    """Load mutated bad PDDL domains from mutated_bad_PDDL_AP/.
    Returns list of {cve_id, description, attack_paths: [{ap_id, domain}]}."""
    bad_dataset = []
    bad_dir = Path(bad_pddl_dir)
    if not bad_dir.exists():
        print(f"WARNING: {bad_pddl_dir} not found")
        return bad_dataset
    for cve_dir in sorted(bad_dir.iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id, "")
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir():
                continue
            domain_file = ap_dir / DOMAIN_FILE
            if domain_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                })
        if attack_paths:
            bad_dataset.append({
                'cve_id': cve_id,
                'description': description,
                'attack_paths': attack_paths,
            })
    return bad_dataset

def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


MODEL_VRAM_GB = {
    # rough fp16/bf16 weight footprint for GPU/CPU decision
    "Qwen3-Embedding-8B": 16.5,
    "Qwen3-Embedding-4B": 8.5,
    "Qwen3-Embedding-0.6B": 1.5,
    "bge-base": 0.5,
    "all-MiniLM": 0.2,
}

def _estimate_vram_gb(model_name):
    for key, gb in MODEL_VRAM_GB.items():
        if key in model_name:
            return gb
    return 2.0  # default safety estimate

def load_embedding_model(model_name, headroom=1.4):
    """Load a SentenceTransformer bi-encoder.
    Pre-check free VRAM: if not enough (estimated * headroom) -> CPU.
    Avoids Windows WDDM silent shared-memory spillover (~10x slower than CPU).
    Always falls back to CPU on OOM.
    """
    TRUST_REMOTE = ["Qwen3-Embedding", "nomic-ai/", "jinaai/jina-embeddings-v3"]
    BF16_ON_CPU = ["Qwen3-Embedding-4B", "Qwen3-Embedding-8B"]
    kwargs = {"trust_remote_code": True} if any(t in model_name for t in TRUST_REMOTE) else {}

    needed = _estimate_vram_gb(model_name) * headroom
    use_gpu = False
    free_gb = 0.0
    if torch.cuda.is_available():
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        if free_gb >= needed:
            use_gpu = True
        else:
            print(f"  [VRAM CHECK] {model_name}: need ~{needed:.1f} GB but only {free_gb:.1f} GB free -> CPU")
    if not use_gpu:
        cpu_kwargs = dict(kwargs)
        if any(t in model_name for t in BF16_ON_CPU):
            cpu_kwargs["model_kwargs"] = {"torch_dtype": torch.bfloat16, "use_safetensors": True}
        else:
            cpu_kwargs["model_kwargs"] = {"use_safetensors": True}
        model = SentenceTransformer(model_name, device="cpu", **cpu_kwargs)
        print(f"  Loaded {model_name} on CPU{' (bf16)' if cpu_kwargs.get('model_kwargs', {}).get('torch_dtype') else ''}")
        return model
    try:
        gpu_kwargs = dict(kwargs)
        gpu_kwargs["model_kwargs"] = {"use_safetensors": True}
        model = SentenceTransformer(model_name, **gpu_kwargs)
        print(f"  Loaded {model_name} on {model.device} (free was {free_gb:.1f} GB)")
        return model
    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        import gc; gc.collect(); torch.cuda.empty_cache()
        oom_kwargs = dict(kwargs)
        oom_kwargs["model_kwargs"] = {"use_safetensors": True}
        model = SentenceTransformer(model_name, device="cpu", **oom_kwargs)
        print(f"  GPU OOM ({type(e).__name__}), loaded {model_name} on CPU")
        return model




def with_gpu_oom_cpu_fallback(model, fn, *args, **kwargs):
    """Run fn(*args, **kwargs). On CUDA OutOfMemoryError, move model to CPU
    and retry once. Use this around encode-heavy calls when a model is loaded
    on GPU but a particular batch (long seq, big batch) might exceed VRAM.
    """
    try:
        return fn(*args, **kwargs)
    except torch.cuda.OutOfMemoryError:
        import gc
        gc.collect(); torch.cuda.empty_cache()
        print(f"  [OOM on GPU] -> moving model to CPU and retrying...")
        try:
            model.to("cpu")
        except Exception as e:
            print(f"  (model.to('cpu') failed: {e})")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return fn(*args, **kwargs)


def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer



def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
bad_dataset = load_bad_dataset(BAD_PDDL_DIR, cve_descriptions)
print(f"Bad (mutated) examples: {len(bad_dataset)} CVEs, {sum(len(e['attack_paths']) for e in bad_dataset)} domains")
prompt_env = load_prompts(PROMPTS_PATH)


embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


def save_cv_record(cv_path, model_name, threshold, fold_thresholds, row, labels, build_seconds, cv_seconds, cv_method="reference_full_matrix"):
    """Build CV record, dedup by model, and save to JSONL."""
    record = {
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "fold_thresholds": fold_thresholds,
        "accuracy": row.get("accuracy", float("nan")),
        "precision": row.get("1__precision", float("nan")),
        "recall": row.get("1__recall", float("nan")),
        "f1": row.get("1__f1-score", float("nan")),
        "tpr": row["tpr"],
        "fpr": row["fpr"],
        "n_positive_pairs": int(labels.sum()),
        "n_negative_pairs": int((labels == 0).sum()),
        "build_pairs_seconds": build_seconds,
        "cv_calibration_seconds": cv_seconds,
    }
    existing = []
    if os.path.exists(cv_path):
        with open(cv_path) as f:
            existing = [json.loads(line) for line in f if line.strip()]
        existing = [r for r in existing if not (r.get("model") == model_name and r.get("cv_method", "reference_full_matrix") == cv_method)]
    existing.append(record)
    with open(cv_path, "w") as f:
        for r in existing:
            f.write(json.dumps(r) + "\n")
    return record


def save_intrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, ap_id, similarity, prediction, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, test_ap, best_ref_ap, similarity, prediction, n_references, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "cve_id": cve_id,
        "test_ap": test_ap,
        "best_ref_ap": best_ref_ap,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "n_references": n_references,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "label": label, "response_length": len(response),
        "parse_success": label is not None,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, scores, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "verdict": verdict, "final_score": final_score, "scores": scores,
        "response_length": len(response),
        "parse_success": "parse_error" not in scores,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "label": label, "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "verdict": verdict, "final_score": final_score,
        "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def prepare_intrinsic_texts(dataset):
    """Extract texts and build desc×domain pair structure (matches build_intrinsic_pairs).
    Returns:
        descs: unique CVE descriptions (one per CVE)
        domains: 55 reference domains (one per attack path)
        domain_cve_ids: 55 CVE IDs (domain-side, kept for backward compatibility)
        labels: 1155 pair labels (1 if same CVE, else 0), iterated descs-major
        groups: 1155 description-side CVE IDs (for GroupKFold)
    """
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    n_d, n_p = len(descs), len(domains)
    labels = np.array([1 if desc_cve_ids[i] == domain_cve_ids[j] else 0
                       for i in range(n_d) for j in range(n_p)])
    groups = np.array([desc_cve_ids[i] for i in range(n_d) for j in range(n_p)])
    return descs, domains, domain_cve_ids, labels, groups


def compute_intrinsic_scores(descs, domains, model):
    """Compute desc×domain similarity scores (flattened, descs-major)."""
    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)
    n_d, n_p = len(descs), len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n_d) for j in range(n_p)])
    return scores


def prepare_extrinsic_texts(dataset):
    """Extract texts and build pair structure for extrinsic (model-independent)."""
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))
    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]
    n = len(all_entries)
    labels = np.array([1 if cve_ids[i] == cve_ids[j] else 0 for i in range(n) for j in range(i+1, n)])
    groups = np.array([cve_ids[i] for i in range(n) for j in range(i+1, n)])
    return domains, cve_ids, labels, groups


def compute_extrinsic_scores(domains, model):
    """Compute similarity scores for pre-prepared extrinsic texts."""
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()
    n = len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n) for j in range(i+1, n)])
    return scores


# ─────────────── Reference pair score logging ───────────────
PAIR_SCORES_PATH = os.path.join(RESULTS_BASE, "semantic", "reference_pair_scores.jsonl")
os.makedirs(os.path.dirname(PAIR_SCORES_PATH), exist_ok=True)

def _append_pair_scores_dedup(file_path, new_records, key_fields=("model", "mode", "cv_method")):
    """Append records to jsonl, but first remove any prior records with the same
    (model, mode, cv_method). Lets us re-run a model cell idempotently."""
    existing = []
    if os.path.exists(file_path):
        with open(file_path) as f:
            for line in f:
                if line.strip():
                    existing.append(json.loads(line))
    new_keys = {tuple(r[k] for k in key_fields) for r in new_records}
    kept = [r for r in existing if tuple(r.get(k) for k in key_fields) not in new_keys]
    with open(file_path, "w") as f:
        for r in kept + new_records:
            f.write(json.dumps(r) + "\n")

def save_pair_scores_3311a(model_name, scores, dataset, file_path=None):
    """Save (desc x reference-domain) pair similarities."""
    file_path = file_path or PAIR_SCORES_PATH
    descs_meta = [e["cve_id"] for e in dataset]
    domains_meta = [(e["cve_id"], ap["ap_id"]) for e in dataset for ap in e["attack_paths"]]
    n_d, n_p = len(descs_meta), len(domains_meta)
    assert len(scores) == n_d * n_p, f"expected {n_d*n_p} scores, got {len(scores)}"
    ts = datetime.now().isoformat()
    records = []
    for i in range(n_d):
        for j in range(n_p):
            records.append({
                "timestamp": ts, "model": model_name,
                "mode": "intrinsic", "cv_method": "reference_full_matrix",
                "pair_kind": "matrix",
                "desc_cve_id": descs_meta[i],
                "domain_cve_id": domains_meta[j][0],
                "domain_ap_id": domains_meta[j][1],
                "domain_source": "reference",
                "bad_difficulty": None,
                "similarity": float(scores[i * n_p + j]),
                "label": 1 if descs_meta[i] == domains_meta[j][0] else 0,
            })
    _append_pair_scores_dedup(file_path, records)
    print(f"  [pair-scores] saved {len(records)} records for {model_name}")

def save_pair_scores_3311c(model_name, scores, dataset, bad_optimized, file_path=None):
    """Save pair similarities for balanced CV (mirrors build_intrinsic_pairs_good_bad order).
    Block 1: positive_good (one per reference AP).
    Block 2: negative_bad (the 20 bad_optimized entries; difficulty tag preserved).
    Block 3: negative_cross_cve (reference AP paired with a random other-CVE description, seed=42)."""
    file_path = file_path or PAIR_SCORES_PATH
    pair_meta = []
    # Block 1
    for entry in dataset:
        for ap in entry["attack_paths"]:
            pair_meta.append({
                "pair_kind": "positive_good",
                "desc_cve_id": entry["cve_id"],
                "domain_cve_id": entry["cve_id"],
                "domain_ap_id": ap["ap_id"],
                "domain_source": "reference",
                "bad_difficulty": None,
                "label": 1,
            })
    # Block 2
    for entry in bad_optimized:
        pair_meta.append({
            "pair_kind": "negative_bad",
            "desc_cve_id": entry["cve_id"],
            "domain_cve_id": entry["cve_id"],
            "domain_ap_id": entry.get("ap_id"),
            "domain_source": "mutated_bad",
            "bad_difficulty": entry.get("difficulty"),
            "label": 0,
        })
    # Block 3 (must use the same RNG seed as build_intrinsic_pairs_good_bad)
    all_entries = [(e["cve_id"], ap["ap_id"]) for e in dataset for ap in e["attack_paths"]]
    rng = np.random.RandomState(42)
    for cve_id, ap_id in all_entries:
        others = [e for e in dataset if e["cve_id"] != cve_id]
        if others:
            other = others[rng.randint(len(others))]
            pair_meta.append({
                "pair_kind": "negative_cross_cve",
                "desc_cve_id": other["cve_id"],
                "domain_cve_id": cve_id,
                "domain_ap_id": ap_id,
                "domain_source": "reference",
                "bad_difficulty": None,
                "label": 0,
            })
    assert len(scores) == len(pair_meta), f"expected {len(pair_meta)} scores, got {len(scores)}"
    ts = datetime.now().isoformat()
    records = []
    for i, meta in enumerate(pair_meta):
        records.append({
            "timestamp": ts, "model": model_name,
            "mode": "intrinsic", "cv_method": "control_hardeasy_bad_ratio",
            **meta, "similarity": float(scores[i]),
        })
    _append_pair_scores_dedup(file_path, records)
    n_pos = sum(1 for r in records if r["pair_kind"] == "positive_good")
    n_bad = sum(1 for r in records if r["pair_kind"] == "negative_bad")
    n_cross = sum(1 for r in records if r["pair_kind"] == "negative_cross_cve")
    print(f"  [pair-scores] saved {len(records)} for {model_name} (pos={n_pos}, bad={n_bad}, cross={n_cross})")

def save_pair_scores_3321a(model_name, scores, dataset, file_path=None):
    """Save C(n,2) upper-triangle (domain x domain) pair similarities."""
    file_path = file_path or PAIR_SCORES_PATH
    domains_meta = [(e["cve_id"], ap["ap_id"]) for e in dataset for ap in e["attack_paths"]]
    n = len(domains_meta)
    expected = n * (n - 1) // 2
    assert len(scores) == expected, f"expected {expected} scores, got {len(scores)}"
    ts = datetime.now().isoformat()
    records = []
    idx = 0
    for i in range(n):
        for j in range(i + 1, n):
            records.append({
                "timestamp": ts, "model": model_name,
                "mode": "extrinsic", "cv_method": "reference_full_matrix",
                "pair_kind": "matrix_upper_tri",
                "i_cve_id": domains_meta[i][0], "i_ap_id": domains_meta[i][1],
                "j_cve_id": domains_meta[j][0], "j_ap_id": domains_meta[j][1],
                "similarity": float(scores[idx]),
                "label": 1 if domains_meta[i][0] == domains_meta[j][0] else 0,
            })
            idx += 1
    _append_pair_scores_dedup(file_path, records)
    print(f"  [pair-scores] saved {len(records)} records for {model_name}")


In [ ]:
# ── Build test samples: ALL reference + ALL bad ──
test_samples = []

# All reference APs
for entry in dataset:
    for ap in entry["attack_paths"]:
        test_samples.append(("reference", entry["cve_id"], ap["ap_id"], ap["domain"]))

# All bad (mutated) APs
for entry in bad_dataset:
    for ap in entry["attack_paths"]:
        test_samples.append(("bad", entry["cve_id"], ap["ap_id"], ap["domain"]))

print(f"Test samples: {len(test_samples)}")
n_ref = sum(1 for s in test_samples if s[0] == "reference")
n_bad = sum(1 for s in test_samples if s[0] == "bad")
print(f"  reference: {n_ref}, bad: {n_bad}")


In [ ]:
def domain_stats(domain_pddl):
    """Extract structural stats from a PDDL domain string."""
    return {
        "domain_size_bytes": len(domain_pddl.encode("utf-8")),
        "n_actions": len(re.findall(r"\(:action\s", domain_pddl)),
        "n_predicates": len(re.findall(r"\([\w-]+", re.findall(r"\(:predicates([^)]*(?:\([^)]*\))*[^)]*?)\)", domain_pddl, re.DOTALL)[0])) if re.findall(r"\(:predicates", domain_pddl) else 0,
        "n_types": len(re.findall(r"\(:types([^)]*?)\)", domain_pddl, re.DOTALL)[0].split()) if re.findall(r"\(:types", domain_pddl) else 0,
    }

In [ ]:
def get_peak_rss_mb():
    """Get current process peak RSS in MB (cross-platform).
    Unix: uses resource.getrusage (macOS: bytes, Linux: KB).
    Windows / fallback: uses psutil if available, else returns None."""
    if resource is not None:
        ru = resource.getrusage(resource.RUSAGE_CHILDREN)
        if platform.system() == "Darwin":
            return round(ru.ru_maxrss / 1024 / 1024, 2)
        return round(ru.ru_maxrss / 1024, 2)
    try:
        import psutil
        return round(psutil.Process().memory_info().rss / 1024 / 1024, 2)
    except ImportError:
        return None


## S1: Syntax Check

In [ ]:
enhsp = create_enhsp_checker()
# ── Run syntax check ──
t_start_syntax = time.time()
reference_syntax_results = []

save_dir = os.path.join(RESULTS_BASE, "syntax")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_syntax.jsonl")

open(results_path, "w").close()

for source, cve_id, ap_id, domain_pddl in test_samples:
    try:
        problem_str = generate_problem(domain_pddl)
    except ValueError as e:
        # e.g. remove_stride_goal mutant has no STRIDE goal
        stats = domain_stats(domain_pddl)
        result = {"timestamp": datetime.now().isoformat(),
            "source": source, "cve_id": cve_id, "ap_id": ap_id,
            "syntax_ok": False, "error": str(e), "elapsed_seconds": 0.0, **stats}
        reference_syntax_results.append(result)
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"[{source}] {cve_id}/{ap_id}  syntax: SKIP ({e})")
        continue
    t0 = time.time()
    r_enhsp = enhsp.check_from_string(domain_pddl, problem_str)
    elapsed = time.time() - t0
    stats = domain_stats(domain_pddl)

    result = {"timestamp": datetime.now().isoformat(),
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "syntax_ok": r_enhsp.success, "error": r_enhsp.error,
        "elapsed_seconds": round(elapsed, 4), **stats}
    reference_syntax_results.append(result)
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    print(f"[{source}] {cve_id}/{ap_id}  syntax: {r_enhsp.success}  {elapsed:.3f}s")

t_syntax = time.time() - t_start_syntax
n_pass = sum(1 for r in reference_syntax_results if r["syntax_ok"])
print(f"Syntax: {n_pass}/{len(reference_syntax_results)} passed, time: {t_syntax:.2f}s")


## S2: Solvability Check

In [ ]:
ff = create_ff_checker()

# ── Run solvability check ──
t_start_solv = time.time()
reference_solvability_results = []

save_dir = os.path.join(RESULTS_BASE, "solvability")
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, "results_solvability.jsonl")

open(results_path, "w").close()

for source, cve_id, ap_id, domain_pddl in test_samples:
    try:
        problem_str = generate_problem(domain_pddl)
    except ValueError as e:
        stats = domain_stats(domain_pddl)
        result = {"timestamp": datetime.now().isoformat(),
            "source": source, "cve_id": cve_id, "ap_id": ap_id,
            "solvable": False, "plan_length": None, "plan_cost": None,
            "plan": None, "error": str(e), "elapsed_seconds": 0.0, **stats}
        reference_solvability_results.append(result)
        with open(results_path, "a") as f:
            f.write(json.dumps(result) + "\n")
        print(f"[{source}] {cve_id}/{ap_id}  SKIP ({e})")
        continue
    t0 = time.time()
    r_ff = ff.check_from_string(domain_pddl, problem_str)
    elapsed = time.time() - t0
    stats = domain_stats(domain_pddl)

    result = {"timestamp": datetime.now().isoformat(),
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "solvable": r_ff.solvable, "plan_length": r_ff.plan_length,
        "plan_cost": r_ff.plan_cost, "plan": r_ff.plan,
        "error": r_ff.error, "elapsed_seconds": round(elapsed, 4), **stats}
    reference_solvability_results.append(result)
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    status = "SOLVABLE" if r_ff.solvable else "FAIL"
    print(f"[{source}] {cve_id}/{ap_id}  {status}  plan_length={r_ff.plan_length}  {elapsed:.3f}s")

t_solv = time.time() - t_start_solv
n_solvable = sum(1 for r in reference_solvability_results if r["solvable"])
print(f"Solvability: {n_solvable}/{len(reference_solvability_results)} solvable, time: {t_solv:.2f}s")


## S3: Semantic Evaluation

### S3.1: Embedding-Based Similarity

#### S3.1.1: Intrinsic

In [ ]:
# Step 1: Build positive/negative pairs
def build_intrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for intrinsic embedding evaluation.
    Uses unique CVE descriptions × reference domains.
    Positive (1): reference domain × same CVE description
    Negative (0): reference domain × different CVE description
    Groups: description-side CVE ID (for GroupKFold)
    """
    # unique descriptions (one per CVE)
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    
    # reference domains
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    
    # Encode and compute similarity matrix
    E_desc = model.encode(descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    E_domain = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E_desc @ E_domain.T).float().cpu().numpy()
    
    # Build pairs from matrix
    scores, labels, groups = [], [], []
    for i in range(len(descs)):
        for j in range(len(domains)):
            scores.append(float(sim_matrix[i, j]))
            labels.append(1 if desc_cve_ids[i] == domain_cve_ids[j] else 0)
            groups.append(desc_cve_ids[i])

    return np.array(scores), np.array(labels), np.array(groups)


t_build = time.time()
pair_scores_a, pair_labels_a, pair_groups_a = build_intrinsic_pairs(dataset, embedding_model)
build_pairs_seconds_a = round(time.time() - t_build, 2)
print(f"  Build pairs time: {build_pairs_seconds_a}s")
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {pair_labels_a.sum()}, Negative pairs: {(pair_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(pair_groups_a))}")

# Global list for cross-model comparison CSV
calibration_embedding_rows = []


In [ ]:
# Step 2: CV calibration
def _find_threshold_pr(y_true, y_score):
    """Return the threshold closest to (1,1) in the precision-recall curve."""
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    distances = np.sqrt((1 - prec[1:]) ** 2 + (1 - rec[1:]) ** 2)
    return float(thr[np.argmin(distances)])

def run_calibration(scores, labels, groups, k=5, n_bootstrap=500, random_state=42):
    """
    CVE-level GroupKFold CV with bootstrap threshold search on each train fold.
    Returns: median_threshold, fold_thresholds, y_pred (OOF), y_true (OOF)
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    groups = np.asarray(groups)
    rng = np.random.RandomState(random_state)

    unique_groups = np.unique(groups)
    n_groups = len(unique_groups)
    actual_k = min(k, n_groups)
    if actual_k < k:
        print(f"Warning: only {n_groups} groups, reducing k from {k} to {actual_k}")

    gkf = GroupKFold(n_splits=actual_k)
    fold_thresholds, y_pred_parts, y_true_parts = [], [], []

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(scores, labels, groups)):
        tr_scores, tr_labels = scores[train_idx], labels[train_idx]
        bt = []
        for _ in range(n_bootstrap):
            idx = rng.choice(len(tr_scores), size=len(tr_scores), replace=True)
            bt.append(_find_threshold_pr(tr_labels[idx], tr_scores[idx]))
        fold_thr = float(np.median(bt))
        fold_thresholds.append(fold_thr)
        y_pred_parts.append((scores[val_idx] >= fold_thr).astype(int))
        y_true_parts.append(labels[val_idx])
        val_groups = np.unique(groups[val_idx])
        print(f"  Fold {fold_i+1}: threshold={fold_thr:.4f}, val CVEs={list(val_groups)}")

    return (
        float(np.median(fold_thresholds)),
        fold_thresholds,
        np.concatenate(y_pred_parts),
        np.concatenate(y_true_parts),
    )


t_cv = time.time()

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# cv_threshold_a, cv_fold_thr_a, cv_pred_a, cv_true_a = run_calibration(
#     pair_scores_a, pair_labels_a, pair_groups_a)
# cv_calibration_seconds_a = round(time.time() - t_cv, 2)
# print(f"CV calibration time: {cv_calibration_seconds_a}s")
# print("Fold thresholds:", [f"{t:.4f}" for t in cv_fold_thr_a])
# print(f"Median threshold: {cv_threshold_a:.4f}")


In [ ]:
# ── Prepare intrinsic texts (shared across all embedding models) ──
intr_descs, intr_domains, intr_cve_ids, intr_labels, intr_groups = prepare_intrinsic_texts(dataset)
print(f"Intrinsic pairs prepared: {len(intr_descs)} unique descs, {len(intr_domains)} domains, {len(intr_labels)} pairs")
print(f"  Positive: {intr_labels.sum()}, Negative: {(intr_labels == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(intr_groups))}")


In [ ]:
# === Embedding Model: BAAI/bge-base-en-v1.5 ===
# 512 max_seq (BERT-base hard limit). bs=32 since attention is small.
if "emb_model_bge_base" not in dir():
    emb_model_bge_base = load_embedding_model("BAAI/bge-base-en-v1.5")
EMB_NAME_bge_base = "BAAI/bge-base-en-v1.5"
print(f"\n--- {EMB_NAME_bge_base} on {emb_model_bge_base.device} ---")

_bs = 32

def _compute_scores_bge_base():
    _E_desc = emb_model_bge_base.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True,
                                        show_progress_bar=True, batch_size=_bs)
    _E_pddl = emb_model_bge_base.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True,
                                        show_progress_bar=True, batch_size=_bs)
    _sim = (_E_desc @ _E_pddl.T).float().cpu().numpy()
    del _E_desc, _E_pddl
    n_d, n_p = len(intr_descs), len(intr_domains)
    return np.array([float(_sim[i, j]) for i in range(n_d) for j in range(n_p)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
threshold_bge_base = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_bge_base and json.loads(l).get("cv_method") == "reference_full_matrix"
)
print(f"  Loaded threshold={threshold_bge_base:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# scores_bge_base = with_gpu_oom_cpu_fallback(emb_model_bge_base, _compute_scores_bge_base)
# build_pairs_seconds_bge_base = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_bge_base}: build_pairs {build_pairs_seconds_bge_base}s")
#
# save_pair_scores_3311a(EMB_NAME_bge_base, scores_bge_base, dataset)
#
# t_cv = time.time()
# threshold_bge_base, fold_thr_bge_base, cv_pred_bge_base, cv_true_bge_base = run_calibration(
#     scores_bge_base, intr_labels, intr_groups)
# cv_seconds_bge_base = round(time.time() - t_cv, 2)
# print(f"  CV calibration {cv_seconds_bge_base}s, threshold={threshold_bge_base:.4f}")
#
# row_bge_base = report_row(cv_true_bge_base, cv_pred_bge_base,
#     metric="embedding", model=EMB_NAME_bge_base, mode="intrinsic", threshold=threshold_bge_base)
# print(f"  TPR={row_bge_base['tpr']:.4f}, FPR={row_bge_base['fpr']:.4f}, F1={row_bge_base.get('1__f1-score', 0):.4f}")
# print(classification_report(cv_true_bge_base, cv_pred_bge_base, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
# cv_rec_bge_base = save_cv_record(cv_path, EMB_NAME_bge_base, threshold_bge_base, fold_thr_bge_base,
#     row_bge_base, intr_labels, build_pairs_seconds_bge_base, cv_seconds_bge_base)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_bge_base, "mode": "intrinsic", "threshold": threshold_bge_base,
#     "precision": cv_rec_bge_base["precision"], "recall": cv_rec_bge_base["recall"],
#     "f1": cv_rec_bge_base["f1"], "tpr": row_bge_base["tpr"], "fpr": row_bge_base["fpr"],
#     "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
# })

results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
def _test_loop_bge_base():
    print(f"\nApplying threshold {threshold_bge_base:.4f} ({EMB_NAME_bge_base}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_bge_base.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_bge_base.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= threshold_bge_base else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_bge_base, threshold_bge_base,
                                         cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
        print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_bge_base, _test_loop_bge_base)

del emb_model_bge_base
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


In [ ]:
import torch
free_gb = torch.cuda.mem_get_info()[0] / 1e9
total_gb = torch.cuda.mem_get_info()[1] / 1e9
print(f"GPU free: {free_gb:.2f} GB / total: {total_gb:.2f} GB")

import gc, torch
for name in list(globals()):
      if name.startswith('emb_model_') or name in ('embedding_model', 'embedding_model_2'):
          del globals()[name]
gc.collect(); gc.collect()
if torch.cuda.is_available():
      torch.cuda.empty_cache()
      print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

In [ ]:
# === Embedding Model: Qwen/Qwen3-Embedding-0.6B ===
# Default settings (no max_seq cap) + GPU OOM fallback to CPU.
if "emb_model_qwen3_emb_06b" not in dir():
    emb_model_qwen3_emb_06b = load_embedding_model("Qwen/Qwen3-Embedding-0.6B")
EMB_NAME_qwen3_emb_06b = "Qwen/Qwen3-Embedding-0.6B"
print(f"\n--- {EMB_NAME_qwen3_emb_06b} on {emb_model_qwen3_emb_06b.device} ---")

_bs = 8

def _compute_scores_qwen3_emb_06b():
    _E_desc = emb_model_qwen3_emb_06b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True,
                           show_progress_bar=True, batch_size=_bs)
    _E_pddl = emb_model_qwen3_emb_06b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True,
                           show_progress_bar=True, batch_size=_bs)
    _sim = (_E_desc @ _E_pddl.T).float().cpu().numpy()
    del _E_desc, _E_pddl
    n_d, n_p = len(intr_descs), len(intr_domains)
    return np.array([float(_sim[i, j]) for i in range(n_d) for j in range(n_p)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
threshold_qwen3_emb_06b = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_qwen3_emb_06b and json.loads(l).get("cv_method") == "reference_full_matrix"
)
print(f"  Loaded threshold={threshold_qwen3_emb_06b:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# scores_qwen3_emb_06b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_06b, _compute_scores_qwen3_emb_06b)
# build_pairs_seconds_qwen3_emb_06b = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_qwen3_emb_06b}: build_pairs {build_pairs_seconds_qwen3_emb_06b}s")
#
# save_pair_scores_3311a(EMB_NAME_qwen3_emb_06b, scores_qwen3_emb_06b, dataset)
#
#
# t_cv = time.time()
# threshold_qwen3_emb_06b, fold_thr_qwen3_emb_06b, cv_pred_qwen3_emb_06b, cv_true_qwen3_emb_06b = run_calibration(
#     scores_qwen3_emb_06b, intr_labels, intr_groups)
# cv_seconds_qwen3_emb_06b = round(time.time() - t_cv, 2)
# print(f"  CV calibration {cv_seconds_qwen3_emb_06b}s, threshold={threshold_qwen3_emb_06b:.4f}")
#
# row_qwen3_emb_06b = report_row(cv_true_qwen3_emb_06b, cv_pred_qwen3_emb_06b,
#     metric="embedding", model=EMB_NAME_qwen3_emb_06b, mode="intrinsic", threshold=threshold_qwen3_emb_06b)
# print(f"  TPR={row_qwen3_emb_06b['tpr']:.4f}, FPR={row_qwen3_emb_06b['fpr']:.4f}, F1={row_qwen3_emb_06b.get('1__f1-score', 0):.4f}")
# print(classification_report(cv_true_qwen3_emb_06b, cv_pred_qwen3_emb_06b, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
# cv_rec_qwen3_emb_06b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_06b, threshold_qwen3_emb_06b, fold_thr_qwen3_emb_06b,
#     row_qwen3_emb_06b, intr_labels, build_pairs_seconds_qwen3_emb_06b, cv_seconds_qwen3_emb_06b)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_qwen3_emb_06b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_06b,
#     "precision": cv_rec_qwen3_emb_06b["precision"], "recall": cv_rec_qwen3_emb_06b["recall"],
#     "f1": cv_rec_qwen3_emb_06b["f1"], "tpr": row_qwen3_emb_06b["tpr"], "fpr": row_qwen3_emb_06b["fpr"],
#     "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
# })

results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
def _test_loop_qwen3_emb_06b():
    print(f"\nApplying threshold {threshold_qwen3_emb_06b:.4f} ({EMB_NAME_qwen3_emb_06b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_06b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_06b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= threshold_qwen3_emb_06b else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_06b, threshold_qwen3_emb_06b,
                                         cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
        print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_06b, _test_loop_qwen3_emb_06b)

del emb_model_qwen3_emb_06b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


In [ ]:
# === Embedding Model: Qwen/Qwen3-Embedding-4B ===
# Default settings (no max_seq cap) + GPU OOM fallback to CPU.
if "emb_model_qwen3_emb_4b" not in dir():
    emb_model_qwen3_emb_4b = load_embedding_model("Qwen/Qwen3-Embedding-4B")
EMB_NAME_qwen3_emb_4b = "Qwen/Qwen3-Embedding-4B"
print(f"\n--- {EMB_NAME_qwen3_emb_4b} on {emb_model_qwen3_emb_4b.device} ---")

_bs = 4

def _compute_scores_qwen3_emb_4b():
    _E_desc = emb_model_qwen3_emb_4b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True,
                           show_progress_bar=True, batch_size=_bs)
    _E_pddl = emb_model_qwen3_emb_4b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True,
                           show_progress_bar=True, batch_size=_bs)
    _sim = (_E_desc @ _E_pddl.T).float().cpu().numpy()
    del _E_desc, _E_pddl
    n_d, n_p = len(intr_descs), len(intr_domains)
    return np.array([float(_sim[i, j]) for i in range(n_d) for j in range(n_p)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
threshold_qwen3_emb_4b = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_qwen3_emb_4b and json.loads(l).get("cv_method") == "reference_full_matrix"
)
print(f"  Loaded threshold={threshold_qwen3_emb_4b:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# scores_qwen3_emb_4b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_4b, _compute_scores_qwen3_emb_4b)
# build_pairs_seconds_qwen3_emb_4b = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_qwen3_emb_4b}: build_pairs {build_pairs_seconds_qwen3_emb_4b}s")
#
# save_pair_scores_3311a(EMB_NAME_qwen3_emb_4b, scores_qwen3_emb_4b, dataset)
#
#
# t_cv = time.time()
# threshold_qwen3_emb_4b, fold_thr_qwen3_emb_4b, cv_pred_qwen3_emb_4b, cv_true_qwen3_emb_4b = run_calibration(
#     scores_qwen3_emb_4b, intr_labels, intr_groups)
# cv_seconds_qwen3_emb_4b = round(time.time() - t_cv, 2)
# print(f"  CV calibration {cv_seconds_qwen3_emb_4b}s, threshold={threshold_qwen3_emb_4b:.4f}")
#
# row_qwen3_emb_4b = report_row(cv_true_qwen3_emb_4b, cv_pred_qwen3_emb_4b,
#     metric="embedding", model=EMB_NAME_qwen3_emb_4b, mode="intrinsic", threshold=threshold_qwen3_emb_4b)
# print(f"  TPR={row_qwen3_emb_4b['tpr']:.4f}, FPR={row_qwen3_emb_4b['fpr']:.4f}, F1={row_qwen3_emb_4b.get('1__f1-score', 0):.4f}")
# print(classification_report(cv_true_qwen3_emb_4b, cv_pred_qwen3_emb_4b, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
# cv_rec_qwen3_emb_4b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_4b, threshold_qwen3_emb_4b, fold_thr_qwen3_emb_4b,
#     row_qwen3_emb_4b, intr_labels, build_pairs_seconds_qwen3_emb_4b, cv_seconds_qwen3_emb_4b)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_qwen3_emb_4b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_4b,
#     "precision": cv_rec_qwen3_emb_4b["precision"], "recall": cv_rec_qwen3_emb_4b["recall"],
#     "f1": cv_rec_qwen3_emb_4b["f1"], "tpr": row_qwen3_emb_4b["tpr"], "fpr": row_qwen3_emb_4b["fpr"],
#     "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
# })

results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
def _test_loop_qwen3_emb_4b():
    print(f"\nApplying threshold {threshold_qwen3_emb_4b:.4f} ({EMB_NAME_qwen3_emb_4b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_4b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_4b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= threshold_qwen3_emb_4b else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_4b, threshold_qwen3_emb_4b,
                                         cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
        print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_4b, _test_loop_qwen3_emb_4b)

del emb_model_qwen3_emb_4b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


In [ ]:
# Enable HF online so Qwen3-Embedding-8B can download (~16 GB) -- not cached locally
import os
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE", "(unset)"))
print("TRANSFORMERS_OFFLINE:", os.environ.get("TRANSFORMERS_OFFLINE", "(unset)"))


In [ ]:
# === Embedding Model: Qwen/Qwen3-Embedding-8B ===
# Default settings (no max_seq cap, batch_size=8) + GPU OOM fallback to CPU for safety.
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"
print(f"\n--- {EMB_NAME_qwen3_emb_8b} on {emb_model_qwen3_emb_8b.device} ---")

_bs = 2

# Step 1: Compute scores (OOM-safe: if GPU encode OOMs, move to CPU and retry)
def _compute_scores_qwen3_emb_8b():
    _E_desc = emb_model_qwen3_emb_8b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True,
                                            show_progress_bar=True, batch_size=_bs)
    _E_pddl = emb_model_qwen3_emb_8b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True,
                                            show_progress_bar=True, batch_size=_bs)
    _sim = (_E_desc @ _E_pddl.T).float().cpu().numpy()
    del _E_desc, _E_pddl
    n_d, n_p = len(intr_descs), len(intr_domains)
    return np.array([float(_sim[i, j]) for i in range(n_d) for j in range(n_p)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
threshold_qwen3_emb_8b = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_qwen3_emb_8b and json.loads(l).get("cv_method") == "reference_full_matrix"
)
print(f"  Loaded threshold={threshold_qwen3_emb_8b:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# scores_qwen3_emb_8b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _compute_scores_qwen3_emb_8b)
# build_pairs_seconds_qwen3_emb_8b = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_qwen3_emb_8b}: build_pairs {build_pairs_seconds_qwen3_emb_8b}s")
#
# # Step 2: CV calibration
# save_pair_scores_3311a(EMB_NAME_qwen3_emb_8b, scores_qwen3_emb_8b, dataset)
#
#
# t_cv = time.time()
# threshold_qwen3_emb_8b, fold_thr_qwen3_emb_8b, cv_pred_qwen3_emb_8b, cv_true_qwen3_emb_8b = run_calibration(
#     scores_qwen3_emb_8b, intr_labels, intr_groups)
# cv_seconds_qwen3_emb_8b = round(time.time() - t_cv, 2)
# print(f"  CV calibration {cv_seconds_qwen3_emb_8b}s, threshold={threshold_qwen3_emb_8b:.4f}")
#
# # Step 3: Performance report + save CV
# row_qwen3_emb_8b = report_row(cv_true_qwen3_emb_8b, cv_pred_qwen3_emb_8b,
#     metric="embedding", model=EMB_NAME_qwen3_emb_8b, mode="intrinsic", threshold=threshold_qwen3_emb_8b)
# print(f"  TPR={row_qwen3_emb_8b['tpr']:.4f}, FPR={row_qwen3_emb_8b['fpr']:.4f}, F1={row_qwen3_emb_8b.get('1__f1-score', 0):.4f}")
# print(classification_report(cv_true_qwen3_emb_8b, cv_pred_qwen3_emb_8b, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
# cv_rec_qwen3_emb_8b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, threshold_qwen3_emb_8b, fold_thr_qwen3_emb_8b,
#     row_qwen3_emb_8b, intr_labels, build_pairs_seconds_qwen3_emb_8b, cv_seconds_qwen3_emb_8b)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_qwen3_emb_8b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_8b,
#     "precision": cv_rec_qwen3_emb_8b["precision"], "recall": cv_rec_qwen3_emb_8b["recall"],
#     "f1": cv_rec_qwen3_emb_8b["f1"], "tpr": row_qwen3_emb_8b["tpr"], "fpr": row_qwen3_emb_8b["fpr"],
#     "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
# })

# Step 4: Test on test_samples (OOM-safe)
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
def _test_loop_qwen3_emb_8b():
    print(f"\nApplying threshold {threshold_qwen3_emb_8b:.4f} ({EMB_NAME_qwen3_emb_8b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_8b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= threshold_qwen3_emb_8b else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, threshold_qwen3_emb_8b,
                                         cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
        print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _test_loop_qwen3_emb_8b)

# Free model + reclaim memory
del emb_model_qwen3_emb_8b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


#### S3.1.2: Extrinsic

In [ ]:
# ── Prepare extrinsic texts (shared across all embedding models) ──
extr_domains, extr_cve_ids, extr_labels, extr_groups = prepare_extrinsic_texts(dataset)
print(f"Extrinsic pairs prepared: {len(extr_domains)} domains, {len(extr_labels)} pairs")
print(f"  Positive: {extr_labels.sum()}, Negative: {(extr_labels == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(extr_groups))}")


In [ ]:
# === Embedding Model: BAAI/bge-base-en-v1.5 ===
if "emb_model_bge_base" not in dir():
    emb_model_bge_base = load_embedding_model("BAAI/bge-base-en-v1.5")
EMB_NAME_bge_base = "BAAI/bge-base-en-v1.5"

_bs = 32

def _compute_ext_scores_bge_base():
    _E = emb_model_bge_base.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True,
                                   show_progress_bar=True, batch_size=_bs)
    _sim_matrix = (_E @ _E.T).float().cpu().numpy()
    n_ext = len(extr_domains)
    return np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_threshold_bge_base = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_bge_base
)
print(f"  Loaded threshold={ext_threshold_bge_base:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# ext_scores_bge_base = with_gpu_oom_cpu_fallback(emb_model_bge_base, _compute_ext_scores_bge_base)
# ext_build_seconds_bge_base = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_bge_base}: build_pairs {ext_build_seconds_bge_base}s")
#
# save_pair_scores_3321a(EMB_NAME_bge_base, ext_scores_bge_base, dataset)
#
# t_cv = time.time()
# ext_threshold_bge_base, ext_fold_thr_bge_base, ext_cv_pred_bge_base, ext_cv_true_bge_base = run_calibration(
#     ext_scores_bge_base, extr_labels, extr_groups)
# ext_cv_seconds_bge_base = round(time.time() - t_cv, 2)
# print(f"  CV calibration {ext_cv_seconds_bge_base}s, threshold={ext_threshold_bge_base:.4f}")
#
# ext_row_bge_base = report_row(ext_cv_true_bge_base, ext_cv_pred_bge_base,
#     metric="embedding", model=EMB_NAME_bge_base, mode="extrinsic", threshold=ext_threshold_bge_base)
# print(f"  TPR={ext_row_bge_base['tpr']:.4f}, FPR={ext_row_bge_base['fpr']:.4f}, F1={ext_row_bge_base.get('1__f1-score', 0):.4f}")
# print(classification_report(ext_cv_true_bge_base, ext_cv_pred_bge_base, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
# ext_cv_rec_bge_base = save_cv_record(cv_path, EMB_NAME_bge_base, ext_threshold_bge_base, ext_fold_thr_bge_base,
#     ext_row_bge_base, extr_labels, ext_build_seconds_bge_base, ext_cv_seconds_bge_base)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_bge_base, "mode": "extrinsic", "threshold": ext_threshold_bge_base,
#     "precision": ext_cv_rec_bge_base["precision"], "recall": ext_cv_rec_bge_base["recall"],
#     "f1": ext_cv_rec_bge_base["f1"], "tpr": ext_row_bge_base["tpr"], "fpr": ext_row_bge_base["fpr"],
#     "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
# })

results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
def _test_loop_ext_bge_base():
    print(f"\nApplying threshold {ext_threshold_bge_base:.4f} ({EMB_NAME_bge_base}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        ref_aps = ref_by_cve.get(cve_id, [])
        if not ref_aps:
            print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
            continue
        t0 = time.time()
        E_test = emb_model_bge_base.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        ref_texts = [ap["domain"] for ap in ref_aps]
        E_ref = emb_model_bge_base.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
        sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
        elapsed = time.time() - t0
        best_sim = float(sims.max())
        best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
        pred = True if best_sim >= ext_threshold_bge_base else False
        for j, ref_ap in enumerate(ref_aps):
            sim = float(sims[j])
            p = True if sim >= ext_threshold_bge_base else False
            print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
        save_extrinsic_similarity_result(results_path, source, EMB_NAME_bge_base, ext_threshold_bge_base, cve_id, ap_id,
            [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
        print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

with_gpu_oom_cpu_fallback(emb_model_bge_base, _test_loop_ext_bge_base)

# ── Full evaluation: all samples ──
def _full_extr_bge_base():
    _run_full_extrinsic_eval(emb_model_bge_base, EMB_NAME_bge_base, ext_threshold_bge_base, "reference_full_matrix", results_path)
with_gpu_oom_cpu_fallback(emb_model_bge_base, _full_extr_bge_base)

del emb_model_bge_base
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


In [ ]:
# === Embedding Model: Qwen/Qwen3-Embedding-0.6B ===
# Default settings (no max_seq cap) + GPU OOM fallback to CPU.
if "emb_model_qwen3_emb_06b" not in dir():
    emb_model_qwen3_emb_06b = load_embedding_model("Qwen/Qwen3-Embedding-0.6B")
EMB_NAME_qwen3_emb_06b = "Qwen/Qwen3-Embedding-0.6B"

_bs = 8

def _compute_ext_scores_qwen3_emb_06b():
    _E = emb_model_qwen3_emb_06b.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True,
                      show_progress_bar=True, batch_size=_bs)
    _sim_matrix = (_E @ _E.T).float().cpu().numpy()
    n_ext = len(extr_domains)
    return np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_threshold_qwen3_emb_06b = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_qwen3_emb_06b
)
print(f"  Loaded threshold={ext_threshold_qwen3_emb_06b:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# ext_scores_qwen3_emb_06b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_06b, _compute_ext_scores_qwen3_emb_06b)
# ext_build_seconds_qwen3_emb_06b = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_qwen3_emb_06b}: build_pairs {ext_build_seconds_qwen3_emb_06b}s")
#
# save_pair_scores_3321a(EMB_NAME_qwen3_emb_06b, ext_scores_qwen3_emb_06b, dataset)
#
#
# t_cv = time.time()
# ext_threshold_qwen3_emb_06b, ext_fold_thr_qwen3_emb_06b, ext_cv_pred_qwen3_emb_06b, ext_cv_true_qwen3_emb_06b = run_calibration(
#     ext_scores_qwen3_emb_06b, extr_labels, extr_groups)
# ext_cv_seconds_qwen3_emb_06b = round(time.time() - t_cv, 2)
# print(f"  CV calibration {ext_cv_seconds_qwen3_emb_06b}s, threshold={ext_threshold_qwen3_emb_06b:.4f}")
#
# ext_row_qwen3_emb_06b = report_row(ext_cv_true_qwen3_emb_06b, ext_cv_pred_qwen3_emb_06b,
#     metric="embedding", model=EMB_NAME_qwen3_emb_06b, mode="extrinsic", threshold=ext_threshold_qwen3_emb_06b)
# print(f"  TPR={ext_row_qwen3_emb_06b['tpr']:.4f}, FPR={ext_row_qwen3_emb_06b['fpr']:.4f}, F1={ext_row_qwen3_emb_06b.get('1__f1-score', 0):.4f}")
# print(classification_report(ext_cv_true_qwen3_emb_06b, ext_cv_pred_qwen3_emb_06b, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
# ext_cv_rec_qwen3_emb_06b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_06b, ext_threshold_qwen3_emb_06b, ext_fold_thr_qwen3_emb_06b,
#     ext_row_qwen3_emb_06b, extr_labels, ext_build_seconds_qwen3_emb_06b, ext_cv_seconds_qwen3_emb_06b)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_qwen3_emb_06b, "mode": "extrinsic", "threshold": ext_threshold_qwen3_emb_06b,
#     "precision": ext_cv_rec_qwen3_emb_06b["precision"], "recall": ext_cv_rec_qwen3_emb_06b["recall"],
#     "f1": ext_cv_rec_qwen3_emb_06b["f1"], "tpr": ext_row_qwen3_emb_06b["tpr"], "fpr": ext_row_qwen3_emb_06b["fpr"],
#     "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
# })

results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
def _test_loop_ext_qwen3_emb_06b():
    print(f"\nApplying threshold {ext_threshold_qwen3_emb_06b:.4f} ({EMB_NAME_qwen3_emb_06b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        ref_aps = ref_by_cve.get(cve_id, [])
        if not ref_aps:
            print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
            continue
        t0 = time.time()
        E_test = emb_model_qwen3_emb_06b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        ref_texts = [ap["domain"] for ap in ref_aps]
        E_ref = emb_model_qwen3_emb_06b.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
        sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
        elapsed = time.time() - t0
        best_sim = float(sims.max())
        best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
        pred = True if best_sim >= ext_threshold_qwen3_emb_06b else False
        for j, ref_ap in enumerate(ref_aps):
            sim = float(sims[j])
            p = True if sim >= ext_threshold_qwen3_emb_06b else False
            print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
        save_extrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_06b, ext_threshold_qwen3_emb_06b, cve_id, ap_id,
            [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
        print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_06b, _test_loop_ext_qwen3_emb_06b)

# ── Full evaluation: all samples ──
def _full_extr_qwen3_emb_06b():
    _run_full_extrinsic_eval(emb_model_qwen3_emb_06b, EMB_NAME_qwen3_emb_06b, ext_threshold_qwen3_emb_06b, "reference_full_matrix", results_path)
with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_06b, _full_extr_qwen3_emb_06b)

del emb_model_qwen3_emb_06b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


In [ ]:
# === Embedding Model: Qwen/Qwen3-Embedding-4B ===
# Default settings (no max_seq cap) + GPU OOM fallback to CPU.
if "emb_model_qwen3_emb_4b" not in dir():
    emb_model_qwen3_emb_4b = load_embedding_model("Qwen/Qwen3-Embedding-4B")
EMB_NAME_qwen3_emb_4b = "Qwen/Qwen3-Embedding-4B"

_bs = 4

def _compute_ext_scores_qwen3_emb_4b():
    _E = emb_model_qwen3_emb_4b.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True,
                      show_progress_bar=True, batch_size=_bs)
    _sim_matrix = (_E @ _E.T).float().cpu().numpy()
    n_ext = len(extr_domains)
    return np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_threshold_qwen3_emb_4b = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_qwen3_emb_4b
)
print(f"  Loaded threshold={ext_threshold_qwen3_emb_4b:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# ext_scores_qwen3_emb_4b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_4b, _compute_ext_scores_qwen3_emb_4b)
# ext_build_seconds_qwen3_emb_4b = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_qwen3_emb_4b}: build_pairs {ext_build_seconds_qwen3_emb_4b}s")
#
# save_pair_scores_3321a(EMB_NAME_qwen3_emb_4b, ext_scores_qwen3_emb_4b, dataset)
#
#
# t_cv = time.time()
# ext_threshold_qwen3_emb_4b, ext_fold_thr_qwen3_emb_4b, ext_cv_pred_qwen3_emb_4b, ext_cv_true_qwen3_emb_4b = run_calibration(
#     ext_scores_qwen3_emb_4b, extr_labels, extr_groups)
# ext_cv_seconds_qwen3_emb_4b = round(time.time() - t_cv, 2)
# print(f"  CV calibration {ext_cv_seconds_qwen3_emb_4b}s, threshold={ext_threshold_qwen3_emb_4b:.4f}")
#
# ext_row_qwen3_emb_4b = report_row(ext_cv_true_qwen3_emb_4b, ext_cv_pred_qwen3_emb_4b,
#     metric="embedding", model=EMB_NAME_qwen3_emb_4b, mode="extrinsic", threshold=ext_threshold_qwen3_emb_4b)
# print(f"  TPR={ext_row_qwen3_emb_4b['tpr']:.4f}, FPR={ext_row_qwen3_emb_4b['fpr']:.4f}, F1={ext_row_qwen3_emb_4b.get('1__f1-score', 0):.4f}")
# print(classification_report(ext_cv_true_qwen3_emb_4b, ext_cv_pred_qwen3_emb_4b, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
# ext_cv_rec_qwen3_emb_4b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_4b, ext_threshold_qwen3_emb_4b, ext_fold_thr_qwen3_emb_4b,
#     ext_row_qwen3_emb_4b, extr_labels, ext_build_seconds_qwen3_emb_4b, ext_cv_seconds_qwen3_emb_4b)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_qwen3_emb_4b, "mode": "extrinsic", "threshold": ext_threshold_qwen3_emb_4b,
#     "precision": ext_cv_rec_qwen3_emb_4b["precision"], "recall": ext_cv_rec_qwen3_emb_4b["recall"],
#     "f1": ext_cv_rec_qwen3_emb_4b["f1"], "tpr": ext_row_qwen3_emb_4b["tpr"], "fpr": ext_row_qwen3_emb_4b["fpr"],
#     "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
# })

results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
def _test_loop_ext_qwen3_emb_4b():
    print(f"\nApplying threshold {ext_threshold_qwen3_emb_4b:.4f} ({EMB_NAME_qwen3_emb_4b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        ref_aps = ref_by_cve.get(cve_id, [])
        if not ref_aps:
            print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
            continue
        t0 = time.time()
        E_test = emb_model_qwen3_emb_4b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        ref_texts = [ap["domain"] for ap in ref_aps]
        E_ref = emb_model_qwen3_emb_4b.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
        sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
        elapsed = time.time() - t0
        best_sim = float(sims.max())
        best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
        pred = True if best_sim >= ext_threshold_qwen3_emb_4b else False
        for j, ref_ap in enumerate(ref_aps):
            sim = float(sims[j])
            p = True if sim >= ext_threshold_qwen3_emb_4b else False
            print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
        save_extrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_4b, ext_threshold_qwen3_emb_4b, cve_id, ap_id,
            [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
        print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_4b, _test_loop_ext_qwen3_emb_4b)

# ── Full evaluation: all samples ──
def _full_extr_qwen3_emb_4b():
    _run_full_extrinsic_eval(emb_model_qwen3_emb_4b, EMB_NAME_qwen3_emb_4b, ext_threshold_qwen3_emb_4b, "reference_full_matrix", results_path)
with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_4b, _full_extr_qwen3_emb_4b)

del emb_model_qwen3_emb_4b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


In [ ]:
# === Embedding Model: Qwen/Qwen3-Embedding-8B ===
# Default settings (no max_seq cap, batch_size=8) + GPU OOM fallback to CPU.
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"

_bs = 2

# Step 1: Compute scores (OOM-safe)
def _compute_ext_scores_qwen3_emb_8b():
    _E = emb_model_qwen3_emb_8b.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True,
                                       show_progress_bar=True, batch_size=_bs)
    _sim_matrix = (_E @ _E.T).float().cpu().numpy()
    n_ext = len(extr_domains)
    return np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])

# ── Load threshold from saved CV record ──
save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_threshold_qwen3_emb_8b = next(
    json.loads(l)["threshold"] for l in open(cv_path)
    if json.loads(l)["model"] == EMB_NAME_qwen3_emb_8b
)
print(f"  Loaded threshold={ext_threshold_qwen3_emb_8b:.4f} from {cv_path}")

# ── [Alternative] Real-time CV threshold calibration (commented out; uses pre-computed threshold above) ──
# t_build = time.time()
# ext_scores_qwen3_emb_8b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _compute_ext_scores_qwen3_emb_8b)
# ext_build_seconds_qwen3_emb_8b = round(time.time() - t_build, 2)
# print(f"{EMB_NAME_qwen3_emb_8b}: build_pairs {ext_build_seconds_qwen3_emb_8b}s")
#
# # Step 2: CV calibration
# save_pair_scores_3321a(EMB_NAME_qwen3_emb_8b, ext_scores_qwen3_emb_8b, dataset)
#
#
# t_cv = time.time()
# ext_threshold_qwen3_emb_8b, ext_fold_thr_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b, ext_cv_true_qwen3_emb_8b = run_calibration(
#     ext_scores_qwen3_emb_8b, extr_labels, extr_groups)
# ext_cv_seconds_qwen3_emb_8b = round(time.time() - t_cv, 2)
# print(f"  CV calibration {ext_cv_seconds_qwen3_emb_8b}s, threshold={ext_threshold_qwen3_emb_8b:.4f}")
#
# # Step 3: Performance report + save CV
# ext_row_qwen3_emb_8b = report_row(ext_cv_true_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b,
#     metric="embedding", model=EMB_NAME_qwen3_emb_8b, mode="extrinsic", threshold=ext_threshold_qwen3_emb_8b)
# print(f"  TPR={ext_row_qwen3_emb_8b['tpr']:.4f}, FPR={ext_row_qwen3_emb_8b['fpr']:.4f}, F1={ext_row_qwen3_emb_8b.get('1__f1-score', 0):.4f}")
# print(classification_report(ext_cv_true_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b, zero_division=0))
#
# save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
# os.makedirs(save_dir, exist_ok=True)
# cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
# ext_cv_rec_qwen3_emb_8b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, ext_threshold_qwen3_emb_8b, ext_fold_thr_qwen3_emb_8b,
#     ext_row_qwen3_emb_8b, extr_labels, ext_build_seconds_qwen3_emb_8b, ext_cv_seconds_qwen3_emb_8b)
#
# calibration_embedding_rows.append({
#     "model": EMB_NAME_qwen3_emb_8b, "mode": "extrinsic", "threshold": ext_threshold_qwen3_emb_8b,
#     "precision": ext_cv_rec_qwen3_emb_8b["precision"], "recall": ext_cv_rec_qwen3_emb_8b["recall"],
#     "f1": ext_cv_rec_qwen3_emb_8b["f1"], "tpr": ext_row_qwen3_emb_8b["tpr"], "fpr": ext_row_qwen3_emb_8b["fpr"],
#     "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
# })

# Step 4: Test on test_samples (OOM-safe)
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
def _test_loop_ext_qwen3_emb_8b():
    print(f"\nApplying threshold {ext_threshold_qwen3_emb_8b:.4f} ({EMB_NAME_qwen3_emb_8b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        ref_aps = ref_by_cve.get(cve_id, [])
        if not ref_aps:
            print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
            continue
        t0 = time.time()
        E_test = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        ref_texts = [ap["domain"] for ap in ref_aps]
        E_ref = emb_model_qwen3_emb_8b.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
        sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
        elapsed = time.time() - t0
        best_sim = float(sims.max())
        best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
        pred = True if best_sim >= ext_threshold_qwen3_emb_8b else False
        for j, ref_ap in enumerate(ref_aps):
            sim = float(sims[j])
            p = True if sim >= ext_threshold_qwen3_emb_8b else False
            print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
        save_extrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, ext_threshold_qwen3_emb_8b, cve_id, ap_id,
            [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
        print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _test_loop_ext_qwen3_emb_8b)

# Free model + reclaim memory
# ── Full evaluation: all samples ──
def _full_extr_qwen3_emb_8b():
    _run_full_extrinsic_eval(emb_model_qwen3_emb_8b, EMB_NAME_qwen3_emb_8b, ext_threshold_qwen3_emb_8b, "reference_full_matrix", results_path)
with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _full_extr_qwen3_emb_8b)

del emb_model_qwen3_emb_8b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


### S3.2: LLM-as-Expert

In [ ]:
# ── Response parsing helpers ──
SCORED_CRITERIA = ['F1','F2','A1','A2','C1','C2','C3','V1','V2','V3','N1','N2']
SCORED_CRITERIA_EXT = SCORED_CRITERIA + ['R1','R2','R3']

def parse_scored_response(response_text):
    """Parse LLM scored response JSON, extract scores."""
    try:
        text = response_text.strip()
        if text.startswith('```'):
            text = text.split('\n', 1)[1]
            text = text.rsplit('```', 1)[0]
        data = json.loads(text)
        return data
    except Exception:
        return None

def domain_min_score(scores, criteria=SCORED_CRITERIA):
    """Return min of valid criteria scores (0-5). None if no valid scores."""
    valid = [scores[k] for k in criteria if isinstance(scores.get(k), (int, float))]
    return min(valid) if valid else None

def parse_binary_response(response_text):
    """Parse binary True/False from LLM response."""
    try:
        text = response_text.strip()
        if text.startswith('```'): text = text.split('\n', 1)[1].rsplit('```', 1)[0]
        raw_label = json.loads(text).get('label', '?')
        return True if str(raw_label).lower() in ('true', '1', 'yes') else False
    except Exception:
        # Fallback: check raw text
        t = response_text.strip().lower()
        if 'true' in t: return True
        if 'false' in t: return False
        return None

print('Parse functions loaded')


In [ ]:
# Unified evaluation template (binary/scored × intrinsic/extrinsic via parameters)
eval_template = prompt_env.get_template("completion.md.jinja")

def llm_eval_intrinsic_binary(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """Binary True/False classification: does the PDDL match the CVE?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
# ── Override save functions to include scores, llm_response, and reference_ap ──

def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, reference_ap, verdict, final_score, scores, response, parse_success, usage, elapsed, cost):
    result = {
        'timestamp': datetime.now().isoformat(),
        'llm_model': llm_model, 'seed': seed, 'temperature': temperature,
        'source': source, 'cve_id': cve_id, 'ap_id': ap_id,
        'reference_ap': reference_ap,
        'verdict': verdict, 'final_score': final_score,
        'scores': scores,
        'parse_success': parse_success,
        'response_length': len(response),
        'llm_response': response,
        'usage': {**usage, 'elapsed_seconds': elapsed, 'cost_usd': cost},
    }
    with open(results_path, 'a') as f:
        f.write(json.dumps(result) + '\n')
    return result

def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, reference_ap, label, response, parse_success, usage, elapsed, cost):
    result = {
        'timestamp': datetime.now().isoformat(),
        'llm_model': llm_model, 'seed': seed, 'temperature': temperature,
        'source': source, 'cve_id': cve_id, 'ap_id': ap_id,
        'reference_ap': reference_ap,
        'label': label,
        'parse_success': parse_success,
        'response_length': len(response),
        'llm_response': response,
        'usage': {**usage, 'elapsed_seconds': elapsed, 'cost_usd': cost},
    }
    with open(results_path, 'a') as f:
        f.write(json.dumps(result) + '\n')
    return result

print('Override save functions loaded (with scores + llm_response + reference_ap)')


In [ ]:
# ── Mutation type to criterion mapping ──
mut_to_criterion = {
    'delete_critical_action': 'C2',
    'delete_random_precondition': 'C1',
    'swap_action_effects': 'A2',
    'replace_stride_goal': 'V3',
    'corrupt_exposure_action': 'V1',
    'merge_consecutive_actions': 'A1',
    'inject_capability_violation': 'F1',
    'replace_with_abstract_action': 'F2',
    'replace_exploitation_mechanism': 'V2',
    'scramble_action_names': 'N1',
    'scramble_predicate_names': 'N2',
}
SCORED_CRITERIA = ['F1','F2','A1','A2','C1','C2','C3','V1','V2','V3','N1','N2']
SCORED_CRITERIA_EXT = SCORED_CRITERIA + ['R1','R2','R3']


#### S3.2.1: Local Models (Intrinsic+Extrinsic)

In [ ]:
# ── LLM Server Client Setup ──
# Switch between ollama and llama.cpp by changing base_url
# ollama:   http://localhost:11434/v1
# llama.cpp: http://localhost:8080/v1

BACKEND = "llama.cpp"  # "ollama" or "llama.cpp"

if BACKEND == "llama.cpp":
    ollama_client = OpenAI(
        api_key='not-needed',
        base_url='http://localhost:8080/v1',
        timeout=600.0,
    )
    test_model = "qwen3-32b"  # llama.cpp ignores model name
else:
    ollama_client = OpenAI(
        api_key='ollama',
        base_url='http://localhost:11434/v1',
        timeout=600.0,
    )
    test_model = 'qwen3:4b-fp16'

# Test connection
try:
    test = ollama_client.chat.completions.create(
        model=test_model,
        messages=[{'role': 'user', 'content': 'Say OK'}],
        max_tokens=5,
        temperature=0,
    )
    print(f'{BACKEND} connection OK: {test.choices[0].message.content}')
except Exception as e:
    print(f'{BACKEND} connection FAILED: {e}')


In [ ]:
# Unified evaluation template (binary/scored × intrinsic/extrinsic via parameters)
eval_template = prompt_env.get_template("completion.md.jinja")

def llm_eval_intrinsic_binary(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """Binary True/False classification: does the PDDL match the CVE?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def llm_eval_intrinsic_scored(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """12-criteria scored evaluation (integer 0-5, violation/non-violation scale)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def llm_eval_extrinsic_binary(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """Binary True/False: does the candidate match the CVE and reference?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def llm_eval_extrinsic_scored(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """16-criteria scored evaluation (12 quality + 4 reference-comparison, integer 0-5)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    prompt += ' /no_think'
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=4096,
        seed=seed,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
# ── Debug: test with /no_think ──
resp = ollama_client.chat.completions.create(
    model='qwen3:4b-fp16',
    messages=[{'role': 'user', 'content': 'Say True or False: is 1+1=2? /no_think'}],
    max_tokens=50,
    temperature=0,
)
msg = resp.choices[0].message
print('content:', repr(msg.content))
print('finish_reason:', resp.choices[0].finish_reason)

# Also test with eval prompt
print('\n=== Test with actual eval prompt ===')
response, usage = llm_eval_intrinsic_binary(
    dataset[0]['cve_id'], dataset[0]['description'],
    dataset[0]['attack_paths'][0]['domain'],
    ollama_client, 'qwen3:4b-fp16', seed=42, n_calibration=2)
print(f'Response length: {len(response)}')
print(f'Response: {repr(response[:500])}')
print(f'Parsed label: {parse_binary_response(response)}')
print(f'Tokens: {usage}')


##### qwen3:4b

In [ ]:
# ── Intrinsic Binary: qwen3:4b ──
MODEL = 'qwen3:4b-fp16'
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Intrinsic Scored: qwen3:4b ──
MODEL = 'qwen3:4b-fp16'
results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Binary: qwen3:4b ──
MODEL = 'qwen3:4b-fp16'
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_binary(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Scored: qwen3:4b ──
MODEL = 'qwen3:4b-fp16'
results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:4b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


##### qwen3:8b

In [ ]:
# ── Intrinsic Binary: qwen3:8b ──
MODEL = 'qwen3:8b-fp16'
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Intrinsic Scored: qwen3:8b ──
MODEL = 'qwen3:8b-fp16'
results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Binary: qwen3:8b ──
MODEL = 'qwen3:8b-fp16'
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_binary(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Scored: qwen3:8b ──
MODEL = 'qwen3:8b-fp16'
results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:8b-fp16 | {count} samples | {time.time()-t_start:.1f}s')


##### qwen3:14b

In [ ]:
# ── Intrinsic Binary: qwen3:14b ──
MODEL = 'qwen3:14b'
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Intrinsic Scored: qwen3:14b ──
MODEL = 'qwen3:14b'
results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Binary: qwen3:14b ──
MODEL = 'qwen3:14b'
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_binary(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Scored: qwen3:14b ──
MODEL = 'qwen3:14b'
results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:14b | {count} samples | {time.time()-t_start:.1f}s')


##### qwen3:32b

In [ ]:
# ── Intrinsic Binary: qwen3:32b ──
MODEL = 'qwen3:32b'
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f'results_intrinsic_binary_{model_short_name(MODEL)}.jsonl')
t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:32b | {count} samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Intrinsic Scored: qwen3:32b (skip already completed) ──
MODEL = 'qwen3:32b'
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, f'results_intrinsic_scored_{model_short_name(MODEL)}.jsonl')

# Load already completed keys
done_keys = set()
if os.path.exists(results_path):
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            d = json.loads(line)
            done_keys.add((d.get("source",""), d.get("cve_id",""), d.get("ap_id","")))
print(f"Already done: {len(done_keys)}, remaining: {110 - len(done_keys)}")

t_start = time.time()
count = 0

print(f'--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        if ("reference", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        if ("bad", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_intrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

elapsed = time.time() - t_start
print(f'\nDone: {count} new results in {elapsed:.1f}s')


In [ ]:
# ── Extrinsic Binary: qwen3:32b (skip already completed) ──
MODEL = 'qwen3:32b'
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, f'results_extrinsic_binary_{model_short_name(MODEL)}.jsonl')

# Load already completed keys
done_keys = set()
if os.path.exists(results_path):
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            d = json.loads(line)
            done_keys.add((d.get("source",""), d.get("cve_id",""), d.get("ap_id","")))
print(f"Already done: {len(done_keys)}")

t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        if ("reference", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_extrinsic_binary(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        if ("bad", cve_id, ap["ap_id"]) in done_keys:
            continue
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_binary(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:32b | {count} new samples | {time.time()-t_start:.1f}s')


In [ ]:
# ── Extrinsic Scored: qwen3:32b (skip already completed) ──
MODEL = 'qwen3:32b'
results_path = os.path.join(save_dir_ext, f'results_extrinsic_scored_{model_short_name(MODEL)}.jsonl')

# Load already completed keys
done_keys = set()
if os.path.exists(results_path):
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            d = json.loads(line)
            done_keys.add((d.get("source",""), d.get("cve_id",""), d.get("ap_id","")))
print(f"Already done: {len(done_keys)}")

t_start = time.time()
count = 0

print(f'--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        if ("reference", entry["cve_id"], ap["ap_id"]) in done_keys:
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lookup = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        if ("bad", cve_id, ap["ap_id"]) in done_keys:
            continue
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lookup.get(source_ap)
        if ref_domain is None:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  SKIP: {source_ap} not found')
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                ollama_client, MODEL, seed=42, n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: qwen3:32b | {count} new samples | {time.time()-t_start:.1f}s')


#### S3.2.2: Online Models

In [ ]:
def llm_eval_intrinsic_scored(cve_id, description, domain, client, model, seed=42, n_calibration=2):
    """12-criteria scored evaluation (integer 0-5, violation/non-violation scale)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "intrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=False,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description, domain=domain,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def llm_eval_extrinsic_binary(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """Binary True/False: does the candidate match the CVE and reference?"""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=True, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


In [ ]:
def llm_eval_extrinsic_scored(cve_id, description, domain_reference, domain_generated, client, model, seed=42, n_calibration=2):
    """16-criteria scored evaluation (12 quality + 4 reference-comparison, integer 0-5)."""
    calibration_data = load_calibration_data(CALIBRATION_DATA_PATH, "extrinsic", exclude_cve=cve_id, n=n_calibration)
    prompt = eval_template.render(
        binary=False, extrinsic=True,
        calibration_data=calibration_data,
        cve_id=cve_id, description=description,
        domain=domain_generated, domain_reference=domain_reference,
    )
    t0 = time.time()
    resp = api_call_with_retry(client.chat.completions.create,
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=4096,
    )
    elapsed = time.time() - t0
    content = resp.choices[0].message.content
    # Qwen3 thinking mode: answer in content, reasoning in model_extra
    if not content and hasattr(resp.choices[0].message, 'model_extra'):
        reasoning = resp.choices[0].message.model_extra.get('reasoning', '')
        content = reasoning  # fallback: use reasoning if content is empty
    usage = {
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
        "elapsed_seconds": round(elapsed, 2),
    } if resp.usage else {"elapsed_seconds": round(elapsed, 2)}
    return content, usage


##### GPT-4.1

In [ ]:
# ── GPT-4.1 Client Setup ──
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    timeout=180.0,
)
MODEL = 'gpt-4.1'

# Test
try:
    test = client.chat.completions.create(model=MODEL, messages=[{'role': 'user', 'content': 'Say OK'}], max_completion_tokens=5, temperature=0.0)
    print(f'Connection OK: {test.choices[0].message.content}')
except Exception as e:
    print(f'FAILED: {e}')


###### Intrinsic Binary

In [ ]:
# ── Intrinsic Binary: GPT-4.1 ──
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, 'results_intrinsic_binary_gpt-4.1.jsonl')
t_start = time.time()
count = 0

print('--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-4.1', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-4.1', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-4.1 intrinsic binary | {count} samples | {time.time()-t_start:.1f}s')


###### Intrinsic Scored

In [ ]:
# ── Intrinsic Scored: GPT-4.1 ──
results_path = os.path.join(save_dir, 'results_intrinsic_scored_gpt-4.1.jsonl')
t_start = time.time()
count = 0

print('--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-4.1', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-4.1', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-4.1 intrinsic scored | {count} samples | {time.time()-t_start:.1f}s')


###### Extrinsic Binary

In [ ]:
# ── Extrinsic Binary: GPT-4.1 ──
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, 'results_extrinsic_binary_gpt-4.1.jsonl')
t_start = time.time()
count = 0

# Build ref lookup
ref_lookup = {}
for entry in dataset:
    for ap in entry['attack_paths']:
        ref_lookup[(entry['cve_id'], ap['ap_id'])] = ap['domain']

print('--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], ap['domain'], client, 'gpt-4.1', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lk = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lk.get(source_ap)
        if ref_domain is None: continue
        try:
            response, usage = llm_eval_extrinsic_binary(cve_id, bad_entry['description'], ref_domain, ap['domain'], client, 'gpt-4.1', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-4.1 extrinsic binary | {count} samples | {time.time()-t_start:.1f}s')


###### Extrinsic Scored

In [ ]:
# ── Extrinsic Scored: GPT-4.1 ──
results_path = os.path.join(save_dir_ext, 'results_extrinsic_scored_gpt-4.1.jsonl')
t_start = time.time()
count = 0

print('--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], ap['domain'], client, 'gpt-4.1', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lk = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lk.get(source_ap)
        if ref_domain is None: continue
        try:
            response, usage = llm_eval_extrinsic_scored(cve_id, bad_entry['description'], ref_domain, ap['domain'], client, 'gpt-4.1', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-4.1 extrinsic scored | {count} samples | {time.time()-t_start:.1f}s')


##### GPT-5.5

In [ ]:
# ── Switch to GPT-5.5 ──
MODEL = 'gpt-5.5'
# Note: GPT-5.5 does not support temperature=0 or seed
print(f'Switched to {MODEL}')


###### Intrinsic Binary

In [ ]:
# ── Intrinsic Binary: GPT-5.5 ──
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, 'results_intrinsic_binary_gpt-5.5.jsonl')
t_start = time.time()
count = 0

print('--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-5.5', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-5.5', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-5.5 intrinsic binary | {count} samples | {time.time()-t_start:.1f}s')


###### Intrinsic Scored

In [ ]:
# ── Intrinsic Scored: GPT-5.5 ──
results_path = os.path.join(save_dir, 'results_intrinsic_scored_gpt-5.5.jsonl')
t_start = time.time()
count = 0

print('--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-5.5', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], client, 'gpt-5.5', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-5.5 intrinsic scored | {count} samples | {time.time()-t_start:.1f}s')


###### Extrinsic Binary

In [ ]:
# ── Extrinsic Binary: GPT-5.5 ──
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, 'results_extrinsic_binary_gpt-5.5.jsonl')
t_start = time.time()
count = 0

# Build ref lookup
ref_lookup = {}
for entry in dataset:
    for ap in entry['attack_paths']:
        ref_lookup[(entry['cve_id'], ap['ap_id'])] = ap['domain']

print('--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], ap['domain'], client, 'gpt-5.5', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lk = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lk.get(source_ap)
        if ref_domain is None: continue
        try:
            response, usage = llm_eval_extrinsic_binary(cve_id, bad_entry['description'], ref_domain, ap['domain'], client, 'gpt-5.5', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-5.5 extrinsic binary | {count} samples | {time.time()-t_start:.1f}s')


###### Extrinsic Scored

In [ ]:
# ── Extrinsic Scored: GPT-5.5 ──
results_path = os.path.join(save_dir_ext, 'results_extrinsic_scored_gpt-5.5.jsonl')
t_start = time.time()
count = 0

print('--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], ap['domain'], client, 'gpt-5.5', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lk = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lk.get(source_ap)
        if ref_domain is None: continue
        try:
            response, usage = llm_eval_extrinsic_scored(cve_id, bad_entry['description'], ref_domain, ap['domain'], client, 'gpt-5.5', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: GPT-5.5 extrinsic scored | {count} samples | {time.time()-t_start:.1f}s')


##### DeepSeek-R1

In [ ]:
# ── DeepSeek-R1 Client Setup ──
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com',
    timeout=300.0,
)
MODEL = 'deepseek-reasoner'

# Test
try:
    test = client.chat.completions.create(model=MODEL, messages=[{'role': 'user', 'content': 'Say OK'}], max_completion_tokens=10)
    print(f'Connection OK: {test.choices[0].message.content}')
except Exception as e:
    print(f'FAILED: {e}')


###### Intrinsic Binary

In [ ]:
# ── Intrinsic Binary: DeepSeek-R1 ──
save_dir = os.path.join(RESULTS_BASE, 'semantic', 'intrinsic', 'llm-as-experts')
os.makedirs(save_dir, exist_ok=True)
results_path = os.path.join(save_dir, 'results_intrinsic_binary_deepseek-reasoner.jsonl')
t_start = time.time()
count = 0

print('--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], client, 'deepseek-reasoner', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], client, 'deepseek-reasoner', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  label={label}  tokens={usage.get("total_tokens", 0)}')
            save_intrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], label, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: DeepSeek-R1 intrinsic binary | {count} samples | {time.time()-t_start:.1f}s')


###### Intrinsic Scored

In [ ]:
# ── Intrinsic Scored: DeepSeek-R1 ──
results_path = os.path.join(save_dir, 'results_intrinsic_scored_deepseek-reasoner.jsonl')
t_start = time.time()
count = 0

print('--- Reference (expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], client, 'deepseek-reasoner', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (expected: False) ---')
for entry in bad_dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_intrinsic_scored(entry['cve_id'], entry['description'], ap['domain'], client, 'deepseek-reasoner', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  verdict={verdict}  min={final_score}')
            save_intrinsic_scored_result(results_path, MODEL, 42, 0.0, 'bad', entry['cve_id'], ap['ap_id'], verdict, final_score, scores, response, usage, 0, 0)
        except Exception as e:
            print(f'  [bad] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: DeepSeek-R1 intrinsic scored | {count} samples | {time.time()-t_start:.1f}s')


###### Extrinsic Binary

In [ ]:
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, 'results_extrinsic_binary_deepseek-reasoner.jsonl')
t_start = time.time()
count = 0

In [ ]:
# ── Extrinsic Binary: DeepSeek-R1 ──
save_dir_ext = os.path.join(RESULTS_BASE, 'semantic', 'extrinsic', 'llm-as-experts')
os.makedirs(save_dir_ext, exist_ok=True)
results_path = os.path.join(save_dir_ext, 'results_extrinsic_binary_deepseek-reasoner.jsonl')
t_start = time.time()
count = 0

# Build ref lookup
ref_lookup = {}
for entry in dataset:
    for ap in entry['attack_paths']:
        ref_lookup[(entry['cve_id'], ap['ap_id'])] = ap['domain']

print('--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        try:
            response, usage = llm_eval_extrinsic_binary(entry['cve_id'], entry['description'], ap['domain'], ap['domain'], client, 'deepseek-reasoner', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], label, response, label is not None, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print('\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lk = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lk.get(source_ap)
        if ref_domain is None: continue
        try:
            response, usage = llm_eval_extrinsic_binary(cve_id, bad_entry['description'], ref_domain, ap['domain'], client, 'deepseek-reasoner', n_calibration=2)
            label = parse_binary_response(response)
            count += 1
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  label={label}')
            save_extrinsic_binary_result(results_path, MODEL, 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, label, response, label is not None, usage, time.time()-t_start, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: DeepSeek-R1 extrinsic binary | {count} samples | {time.time()-t_start:.1f}s')


###### Extrinsic Scored

In [ ]:
# ── Extrinsic Scored: DeepSeek-R1 (resume: skip already done) ──
results_path = os.path.join(save_dir_ext, 'results_extrinsic_scored_deepseek-reasoner.jsonl')

# Load already completed samples
done_keys = set()
if os.path.exists(results_path):
    with open(results_path) as f:
        for line in f:
            if not line.strip(): continue
            r = json.loads(line)
            done_keys.add((r.get('source',''), r.get('cve_id',''), r.get('ap_id','')))
    print(f'Already done: {len(done_keys)} samples, skipping these.')

t_start = time.time()
count = 0
skipped = 0

print('--- Reference (self as ref, expected: True) ---')
for entry in dataset:
    for ap in entry['attack_paths']:
        if ('reference', entry['cve_id'], ap['ap_id']) in done_keys:
            skipped += 1
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                entry['cve_id'], entry['description'], ap['domain'], ap['domain'],
                client, 'deepseek-reasoner', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]} vs self  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, 'deepseek-reasoner', 42, 0.0, 'reference', entry['cve_id'], ap['ap_id'], ap['ap_id'], verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [ref] {entry["cve_id"]}/{ap["ap_id"]}  ERROR: {e}')

print(f'\n--- Bad (source AP as ref, expected: False) ---')
for bad_entry in bad_dataset:
    cve_id = bad_entry['cve_id']
    ref_entry = next((e for e in dataset if e['cve_id'] == cve_id), None)
    if ref_entry is None: continue
    ref_lk = {ap['ap_id']: ap['domain'] for ap in ref_entry['attack_paths']}
    for ap in bad_entry['attack_paths']:
        source_ap = ap['ap_id'].split('_')[0]
        ref_domain = ref_lk.get(source_ap)
        if ref_domain is None: continue
        if ('bad', cve_id, ap['ap_id']) in done_keys:
            skipped += 1
            continue
        try:
            response, usage = llm_eval_extrinsic_scored(
                cve_id, bad_entry['description'], ref_domain, ap['domain'],
                client, 'deepseek-reasoner', n_calibration=2)
            scores = parse_scored_response(response) or {'parse_error': True}
            final_score = domain_min_score(scores, SCORED_CRITERIA_EXT) if 'parse_error' not in scores else None
            verdict = True if final_score is not None and final_score >= 3 else False
            count += 1
            elapsed = time.time() - t_start
            print(f'  [bad] {cve_id}/{ap["ap_id"]} vs {source_ap}  verdict={verdict}  min={final_score}')
            save_extrinsic_scored_result(results_path, 'deepseek-reasoner', 42, 0.0, 'bad', cve_id, ap['ap_id'], source_ap, verdict, final_score, scores, response, 'parse_error' not in scores, usage, elapsed, 0)
        except Exception as e:
            print(f'  [bad] {cve_id}/{ap["ap_id"]}  ERROR: {e}')

print(f'\nDone: DeepSeek-R1 extrinsic scored | {count} new + {skipped} skipped | {time.time()-t_start:.1f}s')
